# Visualización de Predicciones

Selecciona una parcela, un ciclo y una ventana para graficar:
- **EVI crudo** (puntos)
- **EVI suavizado** (línea continua)
- **EVI extrapolado** (línea punteada)

In [9]:
import sys
sys.path.append("..")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output
from config import GPKG_PATH

In [10]:
conn = sqlite3.connect(str(GPKG_PATH))

parcelas = pd.read_sql(
    "SELECT id_parcela FROM parcelas_vigentes ORDER BY id_parcela", conn
)
parcela_ids = parcelas["id_parcela"].tolist()

ciclos = pd.read_sql(
    """SELECT id_ciclo, id_parcela, temporada, sos, eos
       FROM produccion_acumulada_ciclo
       ORDER BY id_parcela, sos""",
    conn,
    parse_dates=["sos", "eos"],
)

conn.close()

print(f"{len(parcela_ids)} parcela(s), {len(ciclos)} ciclo(s) cargados.")

9 parcela(s), 68 ciclo(s) cargados.


In [ ]:
def _label_ciclo(row):
    sos_str = row["sos"].strftime("%Y-%m-%d") if pd.notna(row["sos"]) else "?"
    return f"#{row['id_ciclo']}  {row['temporada']}  (SOS {sos_str})"


def _graficar(parcela, ciclo, ventana):
    conn = sqlite3.connect(str(GPKG_PATH))

    ciclo_info = pd.read_sql(
        "SELECT sos, eos, temporada FROM produccion_acumulada_ciclo WHERE id_ciclo = ?",
        conn, params=(ciclo,), parse_dates=["sos", "eos"],
    )
    if ciclo_info.empty:
        conn.close()
        return
    sos = ciclo_info.iloc[0]["sos"]
    eos = ciclo_info.iloc[0]["eos"]
    temporada = ciclo_info.iloc[0]["temporada"]

    padding = pd.Timedelta(days=30)
    fecha_min = sos - padding
    fecha_max = eos + padding if pd.notna(eos) else sos + pd.Timedelta(days=180)

    crudo = pd.read_sql(
        """SELECT fecha, evi_crudo
           FROM series_diarias_vpm
           WHERE id_parcela = ? AND fecha BETWEEN ? AND ?
             AND evi_crudo IS NOT NULL
           ORDER BY fecha""",
        conn,
        params=(parcela, fecha_min.date().isoformat(), fecha_max.date().isoformat()),
        parse_dates=["fecha"],
    )

    suavizado = pd.read_sql(
        """SELECT fecha, evi
           FROM indices_suavizados
           WHERE id_ciclo = ?
           ORDER BY fecha""",
        conn, params=(ciclo,), parse_dates=["fecha"],
    )

    pred = pd.read_sql(
        "SELECT id_prediccion FROM predicciones_ventana WHERE id_ciclo = ? AND ventana = ?",
        conn, params=(ciclo, ventana),
    )

    extrapolado = pd.DataFrame()
    if not pred.empty:
        id_pred = int(pred.iloc[0]["id_prediccion"])
        extrapolado = pd.read_sql(
            """SELECT fecha, evi_extrapolado
               FROM series_extrapoladas_ventana
               WHERE id_prediccion = ?
               ORDER BY fecha""",
            conn, params=(id_pred,), parse_dates=["fecha"],
        )

    conn.close()

    print(f"  Datos: {len(crudo)} crudo, {len(suavizado)} suavizado, {len(extrapolado)} extrapolado")

    fig, ax = plt.subplots(figsize=(12, 5))

    if len(crudo):
        ax.scatter(crudo["fecha"], crudo["evi_crudo"],
            s=18, color="#1f77b4", alpha=0.7, label="EVI crudo", zorder=3)

    if len(suavizado):
        ax.plot(suavizado["fecha"], suavizado["evi"],
            color="#2ca02c", linewidth=2, label="EVI suavizado", zorder=4)

    if len(extrapolado):
        ax.plot(extrapolado["fecha"], extrapolado["evi_extrapolado"],
            color="#d62728", linewidth=2, linestyle="--",
            label=f"EVI extrapolado ({ventana})", zorder=4)

    ax.axvline(sos, color="gray", linestyle=":", alpha=0.5, label="SOS")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("EVI")
    ax.set_title(f"Parcela {parcela}  |  Ciclo #{ciclo}  ({temporada})  |  {ventana}", fontsize=13)
    ax.legend(loc="best")
    ax.set_ylim(-0.15, 1.05)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


parcela_w = widgets.Dropdown(
    options=[(f"Parcela {pid}", pid) for pid in parcela_ids],
    value=parcela_ids[0] if parcela_ids else None,
    description="Parcela:",
    style={"description_width": "initial"},
)

ciclo_w = widgets.Dropdown(description="Ciclo:", style={"description_width": "initial"})
ventana_w = widgets.Dropdown(description="Ventana:", style={"description_width": "initial"})
out = widgets.Output()


def _renderizar():
    p = parcela_w.value
    c = ciclo_w.value
    v = ventana_w.value
    if p is None or c is None or v is None:
        return
    with out:
        clear_output(wait=True)
        print(f"Parcela={p}, Ciclo={c}, Ventana={v}")
        _graficar(p, c, v)


def _actualizar_ciclos(change):
    pid = change["new"]
    filt = ciclos[ciclos["id_parcela"] == pid]
    opts = [(_label_ciclo(row), int(row["id_ciclo"])) for _, row in filt.iterrows()]
    ciclo_w.options = opts
    if opts:
        ciclo_w.value = opts[0][1]
    else:
        ciclo_w.value = None
        ventana_w.options = []
        ventana_w.value = None


def _actualizar_ventanas(change):
    cid = change["new"]
    if cid is None:
        ventana_w.options = []
        ventana_w.value = None
        return
    conn = sqlite3.connect(str(GPKG_PATH))
    ventanas_df = pd.read_sql(
        "SELECT DISTINCT ventana FROM predicciones_ventana WHERE id_ciclo = ? ORDER BY ventana",
        conn, params=(cid,),
    )
    conn.close()
    opts = [(r["ventana"], r["ventana"]) for _, r in ventanas_df.iterrows()]
    ventana_w.options = opts
    ventana_w.value = opts[0][1] if opts else None


parcela_w.observe(_actualizar_ciclos, names="value")
ciclo_w.observe(_actualizar_ventanas, names="value")

# El ultimo eslabon de la cascada: cuando cambia ventana -> renderizar
ventana_w.observe(lambda _: _renderizar(), names="value")

# Cargar estado inicial (cascada: parcela -> ciclo -> ventana -> render)
if parcela_ids:
    _actualizar_ciclos({"new": parcela_w.value})

In [12]:
ui = widgets.VBox([parcela_w, ciclo_w, ventana_w, out])
display(ui)